In [ ]:
import os
import random
import numpy 
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2,VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout,GlobalAveragePooling2D
from tensorflow.keras.callbacks import ReduceLROnPlateau,EarlyStopping
from sklearn.metrics import classification_report,confusion_matrix
import seaborn as sns

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hariomnagar123/face-mask-detection-dataset")

print("Path to dataset files:", path)

In [ ]:
numpy.random.seed(42)
tf.random.set_seed(42)

In [ ]:
path='/root/.cache/kagglehub/datasets/hariomnagar123/face-mask-detection-dataset/versions/1'
print(os.listdir(path))

In [ ]:
path=path+'/FMD_Dataset'
train_dir=path+'/train'
val_dir=path+'/valid'


In [ ]:
def split_train_for_test(train_dir,test_ratio=0.1):
  test_dir=os.path.join(os.path.dirname(train_dir),'test')
  os.makedirs(test_dir,exist_ok=True)

  classes=os.listdir(train_dir)
  for cls in classes:
    cls_path=os.path.join(train_dir,cls)
    test_cls_path=os.path.join(test_dir,cls)
    os.makedirs(test_cls_path,exist_ok=True)

    images=os.listdir(cls_path)
    num_test=int(len(images)*test_ratio)
    test_images=random.sample(images,num_test)

    for img in test_images:
      os.rename(os.path.join(cls_path,img),os.path.join(test_cls_path,img))
  return test_dir

test_dir=split_train_for_test(train_dir,test_ratio=0.1)
print(test_dir)

In [ ]:
train_datagen=ImageDataGenerator(
    rescale=1./255,
    zoom_range=0.2,
    shear_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)
validation_datagen=ImageDataGenerator(rescale=1./255)
test_datagen=ImageDataGenerator(rescale=1./255)

train_generator=train_datagen.flow_from_directory(
    train_dir,
    target_size=(128,128),
    class_mode='binary',
    batch_size=32
)
validation_generator=validation_datagen.flow_from_directory(
    val_dir,
    batch_size=32,
    target_size=(128,128),
    class_mode='binary',

)

test_generator=test_datagen.flow_from_directory(
    test_dir,target_size=(128,128),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

In [ ]:
early_stop=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)
reduce_lr=ReduceLROnPlateau(monitor='val_loss',factor=0.2,patience=3,verbose=1)

In [ ]:
models={
    'MobileNetV2':MobileNetV2(weights='imagenet',include_top=False,input_shape=(128,128,3)),
    'VGG16':VGG16(weights='imagenet',include_top=False,input_shape=(128,128,3))

}

In [ ]:
histories={}

In [ ]:
for model_name,base_model in models.items():a
  print(f'...training {model_name}')
  base_model.trainable=False

  model=Sequential([
      base_model,
      GlobalAveragePooling2D(),
      Dropout(0.4),
      Dense(128,activation='relu'),
      Dropout(0.3),
      Dense(1,activation='sigmoid')
  ])
  model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
  history=model.fit(
      train_generator,
      steps_per_epoch=len(train_generator),
      epochs=30,
      validation_data=validation_generator,
      validation_steps=len(validation_generator),
      callbacks=[early_stop,reduce_lr],
      verbose=1

  )
  histories[model_name]=history
  test_loss,test_acc=model.evaluate(test_generator)
  print(f' {model_name} |test_loss={test_loss:.4f}|test_accuracy={test_acc:.4f}')

In [ ]:
plt.figure(figsize=(12,5))
for model_name,history in histories.items():
  plt.plot(history.history['val_loss'],label=f'{model_name} val loss')
  plt.title('Validation Accuracy Comparison')
  plt.xlabel('Epochs')
  plt.ylabel('Accuracy')
  plt.legend()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.axhline(test_acc, color='r', linestyle='--', label='Test Accuracy')
plt.title('Accuracy Comparison')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.figure(figsize=(12,5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.axhline(test_loss, color='r', linestyle='--', label='Test Loss')
plt.title('Loss Comparison')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()
